# Human Consistency

In [1]:
import json
import os
from os.path import join
import numpy as np
from tqdm import tqdm
from common import utils, metrics


dataset_root = '/home/young/hdd1/coco-search18/'


/home/young/miniconda3/envs/reward/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# SemSS scores

In [12]:
import gzip

def scanpath2categories(seg_map, scanpath):
    string = []
    xs = scanpath['X']
    ys = scanpath['Y']
    for x,y in zip(xs, ys):
        symbol = str(int(seg_map[int(y), int(x)]))
        string.append(symbol)
    return string

def consistency_SSS(preds,
                    fixations,
                    truncate,
                    segmentation_map_dir,
                    truncate_gt,
                    reduce='mean'):
    results = []
    for scanpath in tqdm(preds):
        is_fv = scanpath['condition'] == 'freeview'
        if is_fv:
            key = 'test-{}-{}'.format(scanpath['condition'], scanpath['name'][:-4])
        else:
            key = 'test-{}-{}-{}'.format(scanpath['condition'], scanpath['task'],
                                         scanpath['name'][:-4])
        strings = fixations[key]
        subj = scanpath['subject']
        with gzip.GzipFile(
                join(segmentation_map_dir, scanpath['name'][:-3] + 'npy.gz'),
                "r") as r:
            segmentation_map = np.load(r, allow_pickle=True)
            r.close()
        pred = scanpath2categories(segmentation_map, scanpath)
        scores = []
        human_scores = []
        for gt_subj, gt in enumerate(strings):
            if len(gt) > 0 and subj != (gt_subj + 1):
                pred = pred[:truncate] if len(pred) > truncate else pred
                if truncate_gt:
                    gt = gt[:truncate] if len(gt) > truncate else gt
                score = metrics.nw_matching(pred, gt)
                scores.append(score)
        result = {}
        result['condition'] = scanpath['condition']
        if not is_fv:
            result['task'] = scanpath['task']
        result['name'] = scanpath['name']
        if reduce == 'mean':
            result['score'] = np.array(scores).mean()
        elif reduce == 'max':
            result['score'] = max(scores)
        else:
            raise NotImplementedError
        results.append(result)
    return results


semSS_strings = np.load(dataset_root + '/semantic_seq_full/test.pkl', allow_pickle=True)


# Target-present
fixation_path = dataset_root + 'coco_search_fixations_512x320_on_target_allvalid.json'
with open(fixation_path) as json_file:
    human_scanpaths = json.load(json_file)
prev = list(filter(lambda x: x['condition'] == 'present' and x['split'] == 'test', human_scanpaths))
prev = list(filter(lambda x: x['fixOnTarget'] or x['condition'] == 'absent', prev))
for sp in prev:
    sp['X'][0] = 256
    sp['Y'][0] = 160
    
human_sss = consistency_SSS(
    prev, semSS_strings, 100, f'{dataset_root}/semantic_seq_full/segmentation_maps', False)
print('semantic sequence score = {:.3f}'.format(np.mean([x['score'] for x in human_sss])))

human_sss_2 = consistency_SSS(
    prev, semSS_strings, 2, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
print('semantic sequence score (2) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_2])))

human_sss_4 = consistency_SSS(
    prev, semSS_strings, 4, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
print('semantic sequence score (4) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_4])))

human_sss_6 = consistency_SSS(
    prev, semSS_strings, 6, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
print('semantic sequence score (6) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_6])))

# Target-absent
fixation_path = dataset_root + 'coco_search_fixations_512x320_on_target_allvalid.json'
with open(fixation_path) as json_file:
    human_scanpaths = json.load(json_file)
prev = list(filter(lambda x: x['condition'] == 'absent' and x['split'] == 'test', human_scanpaths))
prev = list(filter(lambda x: x['fixOnTarget'] or x['condition'] == 'absent', prev))
for sp in prev:
    sp['X'][0] = 256
    sp['Y'][0] = 160
    
human_sss = consistency_SSS(
    prev, semSS_strings, 100, f'{dataset_root}/semantic_seq_full/segmentation_maps', False)
print('semantic sequence score = {:.3f}'.format(np.mean([x['score'] for x in human_sss])))

human_sss_2 = consistency_SSS(
    prev, semSS_strings, 2, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
print('semantic sequence score (2) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_2])))

human_sss_4 = consistency_SSS(
    prev, semSS_strings, 4, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
print('semantic sequence score (4) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_4])))

human_sss_6 = consistency_SSS(
    prev, semSS_strings, 6, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
print('semantic sequence score (6) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_6])))


# # Free-view
# fixation_path = dataset_root + 'coco_freeview_fixations_512x320.json'
# with open(fixation_path) as json_file:
#     human_scanpaths = json.load(json_file)
# prev = list(filter(lambda x: x['split'] == 'test', human_scanpaths))
# for sp in prev:
#     sp['X'][0] = 256
#     sp['Y'][0] = 160
    
# human_sss = consistency_SSS(
#     prev, semSS_strings, 100, f'{dataset_root}/semantic_seq_full/segmentation_maps', False)
# print('semantic sequence score = {:.3f}'.format(np.mean([x['score'] for x in human_sss])))

# human_sss_2 = consistency_SSS(
#     prev, semSS_strings, 4, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
# print('semantic sequence score (4) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_2])))

# human_sss_4 = consistency_SSS(
#     prev, semSS_strings, 8, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
# print('semantic sequence score (8) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_4])))

# human_sss_6 = consistency_SSS(
#     prev, semSS_strings, 16, f'{dataset_root}/semantic_seq_full/segmentation_maps', True)
# print('semantic sequence score (16) = {:.3f}'.format(np.mean([x['score'] for x in human_sss_6])))

100%|██████████████████████████████████████| 5424/5424 [00:07<00:00, 712.11it/s]


semantic sequence score = 0.526


100%|█████████████████████████████████████| 5424/5424 [00:04<00:00, 1195.41it/s]


semantic sequence score (2) = 0.598


100%|██████████████████████████████████████| 5424/5424 [00:06<00:00, 801.87it/s]


...

semantic sequence score (2) = 0.594


100%|██████████████████████████████████████| 6120/6120 [00:09<00:00, 623.29it/s]


semantic sequence score (4) = 0.510


100%|██████████████████████████████████████| 6120/6120 [00:13<00:00, 444.47it/s]

semantic sequence score (6) = 0.459


# SS scores

In [14]:

def consistency_SS(preds, clusters, truncate, truncate_gt, reduce='mean'):
    results = []
    for scanpath in tqdm(preds):
        is_fv = scanpath['condition'] == 'freeview'
        if is_fv:
            key = 'test-{}-{}'.format(scanpath['condition'], scanpath['name'][:-4])
        else:
            key = 'test-{}-{}-{}'.format(scanpath['condition'], scanpath['task'],
                                         scanpath['name'][:-4])
        ms = clusters[key]
        strings = ms['strings']
        cluster = ms['cluster']
        pred = metrics.scanpath2clusters(cluster, scanpath)
        scores = []
        subj = scanpath['subject']
        for gt_subj, gt in strings.items():
            if len(gt) > 0 and subj != gt_subj:
                pred = pred[:truncate] if len(pred) > truncate else pred
                if truncate_gt:
                    gt = gt[:truncate] if len(gt) > truncate else gt
                score = metrics.nw_matching(pred, gt)
                scores.append(score)
        result = {}
        result['condition'] = scanpath['condition']
        if not is_fv:
            result['task'] = scanpath['task']
        result['name'] = scanpath['name']
        if reduce == 'mean':
            result['score'] = np.array(scores).mean()
        elif reduce == 'max':
            result['score'] = max(scores)
        else:
            raise NotImplementedError
        results.append(result)
    return results


test_clusters = np.load(dataset_root + 'clusters.npy', allow_pickle=True).item()
# test_clusters['test-present-sink-000000288161']['cluster'] --> meanshift
# test_clusters['test-present-sink-000000288161']['strings'].items()
# --> test_clusters['test-present-sink-000000288161']['cluster'] --> meanshift


# Target-present
fixation_path = dataset_root + 'coco_search_fixations_512x320_on_target_allvalid.json'
with open(fixation_path) as json_file:
    human_scanpaths = json.load(json_file)
prev = list(filter(lambda x: x['condition'] == 'present' and x['split'] == 'test', human_scanpaths))
prev = list(filter(lambda x: x['fixOnTarget'] or x['condition'] == 'absent', prev))

for sp in prev:
    sp['X'][0] = 256
    sp['Y'][0] = 160

print(len(prev))

human_ss = consistency_SS(prev, test_clusters, 100, False)
print('sequence score = {:.3f}'.format(np.mean([x['score'] for x in human_ss])))

human_ss_2 = consistency_SS(prev, test_clusters, 2, True)
print('sequence score (2) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_2])))

human_ss_4 = consistency_SS(prev, test_clusters, 4, True)
print('sequence score (4) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_4])))

human_ss_6 = consistency_SS(prev, test_clusters, 6, True)
print('sequence score (6) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_6])))


# Target-absent
fixation_path = dataset_root + 'coco_search_fixations_512x320_on_target_allvalid.json'
with open(fixation_path) as json_file:
    human_scanpaths = json.load(json_file)
prev = list(filter(lambda x: x['condition'] == 'absent' and x['split'] == 'test', human_scanpaths))
prev = list(filter(lambda x: x['fixOnTarget'] or x['condition'] == 'absent', prev))
for sp in prev:
    sp['X'][0] = 256
    sp['Y'][0] = 160
print(len(prev))

human_ss = consistency_SS(prev, test_clusters, 100, False)
print('sequence score = {:.3f}'.format(np.mean([x['score'] for x in human_ss])))

human_ss_2 = consistency_SS(prev, test_clusters, 2, True)
print('sequence score (2) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_2])))

human_ss_4 = consistency_SS(prev, test_clusters, 4, True)
print('sequence score (4) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_4])))

human_ss_6 = consistency_SS(prev, test_clusters, 6, True)
print('sequence score (6) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_6])))


# # Free-viewing
# fixation_path = dataset_root + 'coco_freeview_fixations_512x320.json'
# with open(fixation_path) as json_file:
#     human_scanpaths = json.load(json_file)
# prev = list(filter(lambda x: x['split'] == 'test', human_scanpaths))

# for sp in prev:
#     sp['X'][0] = 256
#     sp['Y'][0] = 160
# print(len(prev))

# test_clusters = np.load(dataset_root + 'clusters.npy', allow_pickle=True).item()
# human_ss = consistency_SS(prev, test_clusters, 100, False)
# print('sequence score = {:.3f}'.format(np.mean([x['score'] for x in human_ss])))

# human_ss_2 = consistency_SS(prev, test_clusters, 4, True)
# print('sequence score (4) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_2])))

# human_ss_4 = consistency_SS(prev, test_clusters, 8, True)
# print('sequence score (8) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_4])))

# human_ss_6 = consistency_SS(prev, test_clusters, 16, True)
# print('sequence score (16) = {:.3f}'.format(np.mean([x['score'] for x in human_ss_6])))

5424


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5424/5424 [00:12<00:00, 425.21it/s]


sequence score = 0.500


 23%|█████████████████████████████████▋                                                                                                                  | 1235/5424 [00:02<00:06, 614.37it/s]

KeyboardInterrupt

